# Kaleb Alejandro Aguilar Ávila
## Tarea 1 — Parte 1: Language Modelling
### Attention Recurrent Language Model (mLLMs, CIMAT — Otoño 2026)

Datos: tuits MEX-A3T (`data/mex20_train.txt` para entrenar, `data/mex20_val.txt` para evaluar).

| Punto | Modelo | Código |
|---|---|---|
| 1 | RNN (LSTM) + Self-Attention (Q, K, V) | `rnn_attention_lm.py` |
| 2 | Statistical LM (n-gramas) | `slm.py` |
| 3 | Neural LM (Bengio) | `nlm.py` |
| 4 | Tabla de perplejidades + ablación sin atención | este notebook |
| 5 | Generación de texto (2 ejemplos por modelo) | este notebook |

**LLM utilizado para Vibe Coding:** _(completar)_

## 0. Imports y configuración

In [ ]:
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import torch

from common import (Vocab, load_tweets, train_dev_split, set_seed, get_device,
                    train_lm, perplexity)
import slm
import nlm
import rnn_attention_lm as rlm

SEED = 1111
set_seed(SEED)
device = get_device()
print('device:', device)

os.makedirs('checkpoints', exist_ok=True)

## 1. Datos y vocabulario compartido

Para que las perplejidades sean comparables, **los tres modelos usan el mismo tokenizador (`TweetTokenizer`), el mismo vocabulario y la misma convención**: cada tuit es independiente, se predicen todas sus palabras más `</s>`, y las palabras fuera de vocabulario se mapean a `<unk>`.

Train se divide 90/10 en *train* / *dev*; *dev* sólo se usa para early stopping. **Val se usa únicamente para reportar la perplejidad final.**

In [ ]:
train_full = load_tweets('data/mex20_train.txt')
val = load_tweets('data/mex20_val.txt')
train, dev = train_dev_split(train_full, dev_frac=0.1, seed=SEED)
print(f'train_full={len(train_full)} | train={len(train)} | dev={len(dev)} | val={len(val)}')

VOCAB_SIZE = 5000
vocab = Vocab(max_size=VOCAB_SIZE, min_freq=1, lower=True).fit(train_full)
print('|V| =', len(vocab))
print(f'OOV rate val: {vocab.oov_rate(val):.2%}')
train_full[:5]

## 2. (Punto 2) Statistical Language Model

Trigramas con (a) suavizado add-k / Laplace y (b) interpolación lineal (Jelinek-Mercer).

In [ ]:
results = []

slm_laplace = slm.NgramLM(vocab, n=3, smoothing='laplace', k=1.0).fit(train_full)
ppl = slm_laplace.perplexity(val)
print(f'SLM trigramas Laplace      -> PPL val = {ppl:.2f}')
results.append({'Modelo': 'SLM trigramas (Laplace)', 'PPL val': ppl})

slm_interp = slm.NgramLM(vocab, n=3, smoothing='interpolation', k=0.1,
                         lambdas=[0.1, 0.3, 0.6]).fit(train_full)
ppl = slm_interp.perplexity(val)
print(f'SLM trigramas interpolado  -> PPL val = {ppl:.2f}')
results.append({'Modelo': 'SLM trigramas (interpolación)', 'PPL val': ppl})

(Opcional) Ajuste de las lambdas de interpolación usando **dev** (nunca val):

In [ ]:
best = None
for lambdas in [[0.1, 0.3, 0.6], [0.2, 0.4, 0.4], [0.3, 0.4, 0.3], [0.2, 0.5, 0.3], [0.4, 0.4, 0.2]]:
    m = slm.NgramLM(vocab, n=3, smoothing='interpolation', k=0.1, lambdas=lambdas).fit(train)
    p = m.perplexity(dev)
    print(lambdas, f'dev PPL = {p:.2f}')
    if best is None or p < best[0]:
        best = (p, lambdas)
print('Mejores lambdas:', best[1])

slm_best = slm.NgramLM(vocab, n=3, smoothing='interpolation', k=0.1, lambdas=best[1]).fit(train_full)
ppl = slm_best.perplexity(val)
print(f'SLM interpolado (lambdas ajustadas en dev) -> PPL val = {ppl:.2f}')
results.append({'Modelo': f'SLM trigramas (interp. {best[1]})', 'PPL val': ppl})

**Observaciones (Punto 2):** _(completar)_

## 3. (Punto 3) Neural Language Model de Bengio

In [ ]:
N = 4  # 3 palabras de contexto
nlm_train_loader = nlm.make_loader(train, vocab, N, batch_size=256, shuffle=True)
nlm_dev_loader   = nlm.make_loader(dev,   vocab, N, batch_size=512)
nlm_val_loader   = nlm.make_loader(val,   vocab, N, batch_size=512)

set_seed(SEED)
bengio = nlm.BengioLM(len(vocab), N=N, emb_dim=100, hidden_dim=256, dropout=0.3, pad_id=vocab.pad_id)
t0 = time.time()
hist_nlm = train_lm(bengio, nlm_train_loader, nlm_dev_loader, device, vocab.pad_id,
                    lr=1e-3, num_epochs=30, patience=3, save_path='checkpoints/bengio.pt')
print(f'Tiempo de entrenamiento: {time.time() - t0:.1f}s')

ppl = perplexity(bengio, nlm_val_loader, device, vocab.pad_id)
print(f'NLM Bengio -> PPL val = {ppl:.2f}')
results.append({'Modelo': f'NLM Bengio (N={N})', 'PPL val': ppl})

**Observaciones (Punto 3):** _(completar)_

## 4. (Punto 1) Recurrent Language Model con Self-Attention

- Embeddings (256) → LSTM apilada de 2 capas (512, unidireccional) → **self-attention causal multi-cabeza con Q, K, V** sobre los estados ocultos → `tanh(W_c [h_t; c_t])` → softmax.
- Q = h W_Q, K = h W_K, V = h W_V; pesos = softmax(QKᵀ/√d_k + máscara causal).
- Adam (lr=1e-3), gradient clipping = 1.0, dropout = 0.4, early stopping sobre *dev*.

In [ ]:
RNN_CFG = dict(emb_dim=256, hidden_dim=512, num_layers=2, n_heads=4, dropout=0.4, rnn_type='lstm')
TRAIN_CFG = dict(lr=1e-3, num_epochs=30, patience=3, clip=1.0)

rnn_train_loader = rlm.make_loader(train, vocab, batch_size=64, shuffle=True)
rnn_dev_loader   = rlm.make_loader(dev,   vocab, batch_size=128)
rnn_val_loader   = rlm.make_loader(val,   vocab, batch_size=128)

set_seed(SEED)
rnn_att = rlm.RNNAttentionLM(len(vocab), use_attention=True, pad_id=vocab.pad_id, **RNN_CFG)
print(rnn_att)
print('Parámetros:', rlm.count_parameters(rnn_att))

t0 = time.time()
hist_att = train_lm(rnn_att, rnn_train_loader, rnn_dev_loader, device, vocab.pad_id,
                    save_path='checkpoints/rnn_attention.pt', **TRAIN_CFG)
print(f'Tiempo de entrenamiento: {time.time() - t0:.1f}s')

ppl = perplexity(rnn_att, rnn_val_loader, device, vocab.pad_id)
print(f'RNN + Attention -> PPL val = {ppl:.2f}')
results.append({'Modelo': 'LSTM + Self-Attention (QKV)', 'PPL val': ppl})

### 4.1 Ablación: mismo modelo **sin atención**

In [ ]:
set_seed(SEED)
rnn_noatt = rlm.RNNAttentionLM(len(vocab), use_attention=False, pad_id=vocab.pad_id, **RNN_CFG)
print('Parámetros:', rlm.count_parameters(rnn_noatt))

t0 = time.time()
hist_noatt = train_lm(rnn_noatt, rnn_train_loader, rnn_dev_loader, device, vocab.pad_id,
                      save_path='checkpoints/rnn_no_attention.pt', **TRAIN_CFG)
print(f'Tiempo de entrenamiento: {time.time() - t0:.1f}s')

ppl = perplexity(rnn_noatt, rnn_val_loader, device, vocab.pad_id)
print(f'RNN sin atención -> PPL val = {ppl:.2f}')
results.append({'Modelo': 'LSTM sin atención (ablación)', 'PPL val': ppl})

In [ ]:
plt.figure(figsize=(7, 4))
for name, h in [('Bengio', hist_nlm), ('LSTM+Att', hist_att), ('LSTM', hist_noatt)]:
    plt.plot(range(1, len(h['dev_ppl']) + 1), h['dev_ppl'], marker='o', label=name)
plt.xlabel('Época'); plt.ylabel('PPL dev'); plt.title('Curvas de validación (dev)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 4.2 Visualización de los pesos de atención

In [ ]:
toks, A = rlm.attention_map(rnn_att, vocab, 'la verdad es que no me importa lo que digan', device)
plt.figure(figsize=(6, 5))
plt.imshow(A.numpy(), cmap='viridis')
plt.xticks(range(len(toks)), toks, rotation=90); plt.yticks(range(len(toks)), toks)
plt.xlabel('Key'); plt.ylabel('Query'); plt.colorbar(); plt.title('Self-attention (promedio de cabezas)')
plt.show()

## 5. (Punto 4) Tabla de resultados

In [ ]:
df = pd.DataFrame(results).sort_values('PPL val').reset_index(drop=True)
df['PPL val'] = df['PPL val'].round(2)
df

**Observaciones (Punto 4):** _(completar: ¿se derrotó al SLM y al NLM? ¿qué efecto tuvo quitar la atención?)_

## 6. (Punto 5) Generación de texto — 2 ejemplos por modelo

In [ ]:
prefixes = ['yo creo que', 'la verdad es']
TEMP = 0.8
set_seed(SEED)

for prefix in prefixes:
    print('=' * 80)
    print(f'Prefijo: "{prefix}"')
    print(' SLM         :', slm_best.generate(prefix, max_len=25, temperature=TEMP, seed=SEED))
    print(' NLM Bengio  :', nlm.generate(bengio, vocab, prefix, max_len=25, temperature=TEMP, device=device))
    print(' LSTM + Att  :', rlm.generate(rnn_att, vocab, prefix, max_len=25, temperature=TEMP, device=device))
    print(' LSTM sin Att:', rlm.generate(rnn_noatt, vocab, prefix, max_len=25, temperature=TEMP, device=device))

### Log-likelihood de oraciones (demo)

In [ ]:
oraciones = ['yo creo que sí', 'sí que creo yo', 'no manches me cae mal']
rows = []
for s in oraciones:
    rows.append({'oración': s,
                 'SLM': slm_best.sentence_logprob(s),
                 'NLM': nlm.sentence_logprob(bengio, vocab, s, device),
                 'LSTM+Att': rlm.sentence_logprob(rnn_att, vocab, s, device),
                 'LSTM': rlm.sentence_logprob(rnn_noatt, vocab, s, device)})
pd.DataFrame(rows).round(2)

## 7. Comentario general

_(completar: LLM usado para Vibe Coding, dificultades, alucinaciones del LLM, recursos de cómputo, tiempos, etc.)_